## Data Processing: JKP Global Factor Data (EM Universe)

Preprocesses the JKP EM parquet: coverage filter, per-stock missing filter, cross-sectional rank normalisation to [-0.5, 0.5], median imputation.

### 1. Setup

In [1]:
import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

warnings.filterwarnings('ignore')

raw_path = Path('../data/Global Factor_EM.parquet')
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

coverage_threshold = 0.70
max_miss_frac = 1.0 / 3.0
min_stocks = 30

train_end = '2014-12-31'
val_end = '2019-12-31'

exclude_cols = {
	'id', 'gvkey', 'iid', 'permno', 'permco', 'date', 'eom', 'excntry',
	'size_grp', 'obs_main', 'exch_main', 'common', 'primary_sec',
	'source_crsp', 'comp_tpci', 'crsp_shrcd', 'comp_exchg', 'crsp_exchcd',
	'curcd', 'fx', 'adjfct', 'bidask',
	'ret', 'ret_local', 'ret_exc', 'ret_exc_lead1m',
	'prc', 'prc_local', 'prc_high', 'prc_low',
	'me', 'me_company', 'dolvol', 'shares', 'tvol',
	'ret_lag_dif', 'div_tot',
}

print(f'raw: {raw_path}')
print(f'output: {output_dir}')

raw: ..\data\Global Factor_EM.parquet
output: ..\data\processed


### 2. Load Raw Data and Apply JKP Screens

Read only the columns we need from the parquet in one targeted call to avoid the full 14 GB allocation. Sample screens are applied immediately after loading.

In [2]:
schema = pq.read_schema(raw_path)
all_col_names = schema.names
print(f'total columns in raw parquet: {len(all_col_names)}')

screen_cols = [c for c in ['obs_main', 'exch_main', 'common', 'primary_sec'] if c in all_col_names]
load_always = ['id', 'eom', 'excntry', 'ret_exc_lead1m']

char_candidate = [
	c for c in all_col_names
	if c not in exclude_cols
	and c not in screen_cols
	and c not in load_always
]
print(f'characteristic candidates: {len(char_candidate)}')

needed = load_always + screen_cols + char_candidate
print(f'loading {len(needed)} columns...')

df = pd.read_parquet(raw_path, columns=needed)
df['eom'] = pd.to_datetime(df['eom'])
print(f'loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'date range: {df["eom"].min().date()} to {df["eom"].max().date()}')

n_before = len(df)
screen_mask = pd.Series(True, index=df.index)
for col in screen_cols:
	screen_mask = screen_mask & (df[col] == 1)
df = df[screen_mask][load_always + char_candidate]
print(f'after screens: {len(df):,} rows ({len(df)/n_before*100:.1f}%)')

char_candidate = [
	c for c in char_candidate
	if c in df.columns and pd.api.types.is_numeric_dtype(df[c])
]
print(f'numeric characteristic candidates: {len(char_candidate)}')
gc.collect()

total columns in raw parquet: 443
characteristic candidates: 407
loading 415 columns...
loaded: 4,386,856 rows x 415 columns
date range: 1995-01-31 to 2025-12-31
after screens: 4,386,856 rows (100.0%)
numeric characteristic candidates: 401


2

### 3. Coverage Filter

Select features with at least 70% non-missing observations in the training period. Computed on training data only to prevent lookahead.

In [3]:
df_train_cov = df[df['eom'] <= train_end]
coverage = df_train_cov[char_candidate].notna().mean()
char_cols = sorted([c for c in char_candidate if coverage[c] >= coverage_threshold])
d = len(char_cols)
print(f'features with >={coverage_threshold:.0%} coverage in training period: d={d}')

retained_cov = coverage[char_cols]
print(f'coverage: min={retained_cov.min():.3f} median={retained_cov.median():.3f} max={retained_cov.max():.3f}')

df = df[['id', 'eom', 'excntry', 'ret_exc_lead1m'] + char_cols]
del df_train_cov
gc.collect()
print(f'working dataframe: {df.shape[0]:,} rows x {df.shape[1]} columns')

features with >=70% coverage in training period: d=151
coverage: min=0.701 median=0.777 max=1.000
working dataframe: 4,386,856 rows x 155 columns


### 4. Monthly Preprocessing

For each month: drop firms missing more than one third of characteristics, rank-normalise each characteristic cross-sectionally to [-0.5, 0.5], replace remaining NaN with 0.

In [4]:
monthly_dates = sorted(df['eom'].unique())
print(f'processing {len(monthly_dates)} months...')

processed = []
skipped = 0

for i, eom in enumerate(monthly_dates):
	group = df[df['eom'] == eom]

	n_miss = group[char_cols].isna().sum(axis=1)
	group = group[n_miss <= d * max_miss_frac]

	if len(group) < min_stocks:
		skipped += 1
		continue

	group = group.copy()
	ranks = group[char_cols].rank(pct=True, axis=0) - 0.5
	ranks = ranks.fillna(0.0)
	group[char_cols] = ranks

	processed.append(group)

	if (i + 1) % 60 == 0:
		print(f'  {i+1}/{len(monthly_dates)} months processed')

df_processed = pd.concat(processed, ignore_index=True)
print(f'done: {len(df_processed):,} rows, {df_processed["eom"].nunique()} months retained, {skipped} skipped')
del df, processed
gc.collect()

processing 372 months...
  60/372 months processed
  120/372 months processed
  180/372 months processed
  240/372 months processed
  300/372 months processed
  360/372 months processed
done: 3,758,613 rows, 372 months retained, 0 skipped


0

### 5. Train / Val / Test Split

In [5]:
train_mask = df_processed['eom'] <= train_end
val_mask = (df_processed['eom'] > train_end) & (df_processed['eom'] <= val_end)
test_mask = df_processed['eom'] > val_end

df_train = df_processed[train_mask].copy()
df_val = df_processed[val_mask].copy()
df_test = df_processed[test_mask].copy()

print(f'train: {len(df_train):,} rows, {df_train["eom"].nunique()} months '
	  f'({df_train["eom"].min().date()} to {df_train["eom"].max().date()})')
print(f'val:   {len(df_val):,} rows, {df_val["eom"].nunique()} months '
	  f'({df_val["eom"].min().date()} to {df_val["eom"].max().date()})')
print(f'test:  {len(df_test):,} rows, {df_test["eom"].nunique()} months '
	  f'({df_test["eom"].min().date()} to {df_test["eom"].max().date()})')

train: 1,553,371 rows, 240 months (1995-01-31 to 2014-12-31)
val:   892,993 rows, 60 months (2015-01-31 to 2019-12-31)
test:  1,312,249 rows, 72 months (2020-01-31 to 2025-12-31)


### 6. Save

In [6]:
df_train.to_parquet(output_dir / 'train.parquet', index=False)
df_val.to_parquet(output_dir / 'val.parquet', index=False)
df_test.to_parquet(output_dir / 'test.parquet', index=False)
print(f'saved train/val/test parquets')

all_proc = pd.concat([df_train, df_val, df_test], ignore_index=True)
country_codes = sorted(all_proc['excntry'].dropna().unique().tolist())
country_to_id = {c: i for i, c in enumerate(country_codes)}

country_lookup = all_proc[['id', 'eom', 'excntry']].copy()
country_lookup['country_id'] = country_lookup['excntry'].map(country_to_id).astype('Int16')
country_lookup = country_lookup[['id', 'eom', 'country_id']].dropna()
country_lookup.to_parquet(output_dir / 'country_lookup.parquet', index=False)
print(f'saved country_lookup.parquet ({len(country_lookup):,} rows, {len(country_codes)} countries)')

metadata = {
	'char_cols': char_cols,
	'd': d,
	'coverage_threshold': coverage_threshold,
	'train_end': train_end,
	'val_end': val_end,
	'country_to_id': country_to_id,
	'country_codes': country_codes,
}
with open(output_dir / 'feature_metadata.json', 'w') as f:
	json.dump(metadata, f, indent=2)
print(f'saved feature_metadata.json (d={d})')

saved train/val/test parquets
saved country_lookup.parquet (3,758,613 rows, 24 countries)
saved feature_metadata.json (d=151)


### 7. Sanity Checks

In [7]:
sample_eom = df_train['eom'].iloc[100]
sample = df_train[df_train['eom'] == sample_eom][char_cols]
print(f'sample month {sample_eom.date()}: {len(sample)} firms')
print(f'feature range: [{sample.min().min():.4f}, {sample.max().max():.4f}] (expect approx [-0.5, 0.5])')
print(f'feature mean:  {sample.mean().mean():.6f} (expect approx 0)')
print(f'nan count:     {sample.isna().sum().sum()} (expect 0)')

ret_miss = df_train['ret_exc_lead1m'].isna().mean()
print(f'ret_exc_lead1m missing rate in train: {ret_miss:.2%}')

monthly_n = df_processed.groupby('eom').size()
print(f'monthly cross-section: min={monthly_n.min()}, mean={monthly_n.mean():.0f}, max={monthly_n.max()}')

del df_train, df_val, df_test, df_processed
gc.collect()
print('data processing complete.')

sample month 1995-01-31: 518 firms
feature range: [-0.4981, 0.5000] (expect approx [-0.5, 0.5])
feature mean:  0.000965 (expect approx 0)
nan count:     0 (expect 0)
ret_exc_lead1m missing rate in train: 1.02%
monthly cross-section: min=517, mean=10104, max=20044
data processing complete.
